<a href="https://colab.research.google.com/github/an-311/ResNet/blob/main/ResNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**ResNet Convolutional Neural Network**: Deep CNN with residual connection(skip connections). The main idea behind residual connections is that the network doesnt have to learn complete mapping/transformation of the input to a particular convolution layer but just the additional change to the input brought about by the layer and then the input is added back to the change.So, mathematically speaking, instead of learning this:

                      H(x) = F(x)

We learn the change to the input F(x) and then add the input back:

                      H(x) = F(x) + x   [helps network learn identity]

This helps to solve a major problem seen with training very deep neural networks. Training deeper networks lead to degradation in performance which goes against the intuition that deeper networks should have better and more refined learning than shallower ones. The major reason behind this is due to the problem of vanishing and exploding gradients the deeper networks are often hard to optimize. So, by adding skip connections we can offer an alternative path for the gradient to flow during optimization/training, which helps very deep networks learn and optimize efficiently.

ResNet Architectural Details: ResNet consists of 5 groups of convolutional layers with skip connections enabled in every group and downsampling either through pooling or via strided convolution for each layer, global average pooling and a fully connected layer at the end with softmax to provide final probablity vector for categorization. No of layers engulfed under a skip connection form the residual block.

Here, in the given code we have tried to explore the functioning of ResNet-18 (18 layers) on the ImageNet dataset for image recognition.
**Note: If images are small like in CIFAR-10, aggressive downsampling can destroy useful information.



**Imports**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

**Device**

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

**Transforms**

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

**Datasets & Loaders**

In [ ]:
train_dataset = datasets.ImageFolder("dataset/train", transform=transform)
val_dataset   = datasets.ImageFolder("dataset/val", transform=transform)
test_dataset  = datasets.ImageFolder("dataset/test", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

**Load ResNet-18 and Modify Final Layer**

In [ ]:
model = models.resnet18(pretrained=True)

num_classes = len(train_dataset.classes)
model.fc = nn.Linear(model.fc.in_features, num_classes)

model = model.to(device)

**Loss and Optimizer**

In [ ]:
criterion = nn.CrossEntropyLoss() optimizer = optim.Adam(model.parameters(), lr=0.0001)

**Training Function**

In [ ]:
def evaluate(loader):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    return running_loss / len(loader), correct / total

**Training Loop with Metric Tracking**

In [ ]:
epochs = 15

train_losses, val_losses, test_losses = [], [], []
train_accs, val_accs, test_accs = [], [], []

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / len(train_loader)
    train_acc = correct / total

    val_loss, val_acc = evaluate(val_loader)
    test_loss, test_acc = evaluate(test_loader)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    test_losses.append(test_loss)

    train_accs.append(train_acc)
    val_accs.append(val_acc)
    test_accs.append(test_acc)

    print(f"Epoch [{epoch+1}/{epochs}]")
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f}, Val   Acc: {val_acc:.4f}")
    print(f"Test  Loss: {test_loss:.4f}, Test  Acc: {test_acc:.4f}")
    print("-"*40)

**Plotting Graphs**

In [ ]:
epochs_range = range(1, epochs + 1)

plt.figure(figsize=(12, 10))

# Loss plots
plt.subplot(2, 2, 1)
plt.plot(epochs_range, train_losses, label='Train Loss')
plt.plot(epochs_range, val_losses, label='Val Loss')
plt.plot(epochs_range, test_losses, label='Test Loss')
plt.legend()
plt.title('Loss vs Epochs')

# Accuracy plots
plt.subplot(2, 2, 2)
plt.plot(epochs_range, train_accs, label='Train Acc')
plt.plot(epochs_range, val_accs, label='Val Acc')
plt.plot(epochs_range, test_accs, label='Test Acc')
plt.legend()
plt.title('Accuracy vs Epochs')

plt.tight_layout()
plt.show()